# 🚀 TASK 4.1: CHIẾN DỊCH HUẤN LUYỆN GIAI ĐOẠN 1 & 2 TRÊN GPU CAO CẤP (COLAB PRO)
### Huấn Luyện 5 Epoch Đầu (Curriculum Degradation + DualStreamFocalLoss + Dynamic Gating)

> **Người thực hiện**: Thành viên C *(Training & Evaluation Lead)*  
> **Môi trường thực hiện**: ☁️ **Google Colab GPU Cao Cấp** (Mức tiêu thụ 8.9 Compute Units/giờ - Specs vượt trội A100 SXM4)  
> **Tài nguyên kế thừa (Rule 4)**:  
> - Tập dữ liệu Staging: `/content/drive/MyDrive/Fatformer/datasets/diffusion_staging.tar` (**1.80 GB**, 36.000 ảnh tỷ lệ 90/10 ProGAN / GenImage - Task 3.1)  
> - Kiến trúc tích hợp: SRM 3-Kernels + Dynamic Frequency Gating $\lambda(x) \in [0.0, 2.0]$ + DualStreamFocalLoss (Task 3.3)  
> - Pipeline đã xác thực: Smoke Test Gate 100% PASS (Task 3.6), `train.py` độc lập tối ưu AMP FP16.  
> **Đầu ra nghiệm thu (DoD)**:  
> - Checkpoint trung gian: `/content/drive/MyDrive/Fatformer/checkpoint/fatformer_srm_phase2.pth` lưu trực tiếp trên Google Drive 5TB.  
> - Nhật ký huấn luyện (Loss total, visual, align) qua từng epoch không gặp hiện tượng phân kỳ hay sốc gradient.  

---

### 📌 QUY TRÌNH HUẤN LUYỆN GIAI ĐOẠN 1 & 2 (5 EPOCHS)
1. **Giai đoạn 1 (Epoch 1–2 - Ổn định hóa)**: Khởi tạo êm dịu các adapter SRM và mạng gating với nhiễu nhẹ ($Q \in [70, 90]$), bảo toàn 100% biểu diễn ngữ nghĩa của CLIP ViT-L/14 gốc.
2. **Giai đoạn 2 (Epoch 3–5 - Chuyển tiếp thích ứng)**: Nhiễu nén vừa ($Q \in [45, 70]$), mạng gating $\lambda(x)$ học cách hạ tỷ trọng tần số không gian khi phát hiện khối JPEG vỡ, kích hoạt cơ chế bù trừ từ nhánh vi sai SRM.

In [ ]:
# ================================================================================
# BƯỚC 1: KẾT NỐI GOOGLE DRIVE 5TB & ĐỒNG BỘ MÃ NGUỒN TỪ BRANCH HOANG
# ================================================================================
from google.colab import drive
import os, sys, shutil

print("[*] Đang kết nối Google Drive 5TB...")
drive.mount('/content/drive')
DRIVE_ROOT = "/content/drive/MyDrive/Fatformer"
assert os.path.exists(DRIVE_ROOT), f"❌ LỖI: Không tìm thấy thư mục {DRIVE_ROOT}. Vui lòng kiểm tra lại cấu trúc Google Drive!"
print(f"[✓] Đã kết nối thành công kho lưu trữ Drive 5TB: {DRIVE_ROOT}")

# Đồng bộ mã nguồn Git nhánh hoang
%cd /content
REPO_DIR = "/content/fatformer-xla"
if not os.path.exists(REPO_DIR):
    print("[*] Đang clone repository từ branch hoang...")
    !git clone -b hoang https://github.com/LeeVHoangtk3/fatformer-xla.git
else:
    print("[*] Đang cập nhật mã nguồn mới nhất từ branch hoang...")
    %cd {REPO_DIR}
    !git checkout hoang
    !git pull origin hoang

%cd {REPO_DIR}
print(f"[✓] Thư mục làm việc hiện tại: {os.getcwd()}")

# Cài đặt các gói phụ thuộc cần thiết
!pip install -q ftfy regex tqdm tabulate matplotlib seaborn

In [ ]:
# ================================================================================
# BƯỚC 2: XÁC THỰC MÔI TRƯỜNG PHẦN CỨNG GPU CAO CẤP (8.9 CU/GIỜ)
# ================================================================================
import torch

assert torch.cuda.is_available(), "❌ LỖI: Runtime chưa được cấp phát GPU! Vào Runtime -> Change runtime type -> Chọn GPU cao cấp."
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
gpu_cc = torch.cuda.get_device_capability(0)

print("=" * 70)
print(f"[✓] PHẦN CỨNG GPU PHÁT HIỆN: {gpu_name}")
print(f"  • Tổng dung lượng VRAM: {gpu_mem:.2f} GB")
print(f"  • Compute Capability: {gpu_cc[0]}.{gpu_cc[1]}")
print(f"  • Mức tiêu hao năng lực: ~8.9 Compute Units / giờ")
print(f"  • Ước tính thời gian chạy 5 Epochs: ~20 - 25 phút (~3.0 - 3.7 CU)")
print("=" * 70)

!nvidia-smi

In [ ]:
# ================================================================================
# BƯỚC 3: KIỂM TRA DỮ LIỆU TẬP HUẤN STAGING & THƯ MỤC LƯU CHECKPOINT
# ================================================================================
DATA_TAR = f"{DRIVE_ROOT}/datasets/diffusion_staging.tar"
CKPT_OUT_DIR = f"{DRIVE_ROOT}/checkpoint"

assert os.path.exists(DATA_TAR), f"❌ LỖI: Không tìm thấy tệp {DATA_TAR} trên Google Drive (Task 3.1)!"
tar_size_gb = os.path.getsize(DATA_TAR) / (1024**3)
print(f"[✓] Đã tìm thấy tệp staging: {DATA_TAR}")
print(f"  • Kích thước tệp: {tar_size_gb:.2f} GB (Chuẩn 1.80 GB với 36.000 ảnh 90/10)")

os.makedirs(CKPT_OUT_DIR, exist_ok=True)
print(f"[✓] Thư mục lưu checkpoint trên Drive 5TB: {CKPT_OUT_DIR}")

### 2. KÍCH HOẠT HUẤN LUYỆN 5 EPOCH TASK 4.1 (GIAI ĐOẠN 1 & 2)
- Script chuẩn hóa `train.py` sẽ tự động:
  1. Giải nén `diffusion_staging.tar` sang SSD NVMe cực nhanh của Colab (`/content/dataset_local`).
  2. Khởi tạo mô hình FatFormer với 3 SRM Kernels + Dynamic Frequency Gating.
  3. Đóng băng 94.1% Backbone ViT gốc, chỉ cập nhật 5.9% tham số Adapter.
  4. Huấn luyện với Mixed Precision (AMP FP16) + Gradient Accumulation 2 (Effective Batch Size = 64).
  5. Tự động lưu checkpoint và sao lưu an toàn về Google Drive sau mỗi epoch.

In [ ]:
# ================================================================================
# BƯỚC 4: THỰC THI CHIẾN DỊCH HUẤN LUYỆN CHÍNH THỨC (EPOCHS 1 ĐẾN 5)
# ================================================================================
!python train.py \
    --data_tar /content/drive/MyDrive/Fatformer/datasets/diffusion_staging.tar \
    --epochs 5 \
    --batch_size 32 \
    --grad_accum 2 \
    --lr 1e-4 \
    --lr_gating 1e-3 \
    --use_srm True \
    --use_gating True \
    --use_curriculum True \
    --loss_type focal \
    --output_dir /content/drive/MyDrive/Fatformer/checkpoint \
    --checkpoint_name fatformer_srm_phase2.pth

### 3. KIỂM THỬ NGHIỆM THU CHECKPOINT GIAI ĐOẠN 2 (DoD AUDIT)
Xác thực checkpoint `fatformer_srm_phase2.pth` đã được lưu thành công trên Google Drive 5TB, đảm bảo đầy đủ trọng số mô hình, trạng thái optimizer và GradScaler để sẵn sàng chuyển tiếp sang Task 4.2.

In [ ]:
# ================================================================================
# BƯỚC 5: XÁC THỰC CHECKPOINT TRUNG GIAN FATORMER_SRM_PHASE2.PTH
# ================================================================================
import os, torch

PHASE2_CKPT = f"{DRIVE_ROOT}/checkpoint/fatformer_srm_phase2.pth"
assert os.path.exists(PHASE2_CKPT), f"❌ LỖI: Chưa tìm thấy checkpoint tại {PHASE2_CKPT}!"

ckpt_size_mb = os.path.getsize(PHASE2_CKPT) / (1024**2)
print(f"[✓] Phát hiện checkpoint trung gian: {PHASE2_CKPT}")
print(f"  • Kích thước: {ckpt_size_mb:.2f} MB")

# Kiểm tra tính toàn vẹn của state dict
ckpt_data = torch.load(PHASE2_CKPT, map_location="cpu")
saved_epoch = ckpt_data.get("epoch", None)
print(f"  • Epoch đã lưu trong checkpoint: {saved_epoch}")
assert saved_epoch == 5, f"❌ Lỗi: Kỳ vọng epoch 5 nhưng nhận {saved_epoch}!"

keys_in_ckpt = list(ckpt_data.keys())
print(f"  • Các khóa thành phần có trong checkpoint: {keys_in_ckpt}")
assert "model_state_dict" in ckpt_data, "Thiếu model_state_dict!"
assert "optimizer_state_dict" in ckpt_data, "Thiếu optimizer_state_dict!"
assert "scaler_state_dict" in ckpt_data, "Thiếu scaler_state_dict!"

print("\n" + "=" * 70)
print("🎉 [PASS] TASK 4.1 ĐÃ ĐẠT 100% TIÊU CHUẨN NGHIỆM THU (DoD)!")
print("  • 5 Epochs Giai đoạn 1 & 2 hoàn tất thành công.")
print("  • Checkpoint fatformer_srm_phase2.pth an toàn trên Google Drive 5TB.")
print("  • Sẵn sàng tiến hành Task 4.2 (Epochs 6-8 Hardening) hoặc đánh giá Fast-Eval!")
print("=" * 70)